Calvin Catalog Search

Step 1: Get an LLM running

https://ollama.com/library/gemma3:1b

### Option 1: Run locally:

1. Install `ollama`.
2. Start the server: `ollama serve`.
3. Pull the model: `ollama pull gemma3:1b-it-qat`
  - If you have a lot of memory, or a good GPU, you can try `gemma3:4b-it-qat`.

If you do this, you can use:

```python
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
```

### Option 2: Use the Google Gemini API (talking the OpenAI protocol)

See the instrucitons in CS 375 Homework 1.

## Warm-Up: Structured Outputs

We want to ask the model to output a set of search queries. So we'll want to have the model output in a structured way that's easy for us to parse.

See https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat for an intro.

https://ollama.com/blog/structured-outputs



In [22]:
from pydantic import BaseModel, Field
import openai

First we'll use Pydantic to declare what type of output we want. Despite being named "model", this isn't an AI model; it's basically a data class (a struct).

In [33]:
from pydantic import BaseModel

class SearchTool(BaseModel):
    thinking: str
    queries: list[str]

example_search = SearchTool(
    thinking="The user wants to know some trivia.",
    queries=[
        "What is the capital of France?",
        "What is the largest mammal?",
    ])
example_search

SearchTool(thinking='The user wants to know some trivia.', queries=['What is the capital of France?', 'What is the largest mammal?'])

Now let's ask the LLM to create these queries for us. We'll use `response_format` in the OpenAI API.

We'll need to do some prompt engineering to get reasonable results from this small model.

https://platform.openai.com/docs/guides/text?api-mode=chat

In [39]:
client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL_NAME = "gemma3:1b-it-qat" # change this if you want to use a different model

completion = client.beta.chat.completions.parse(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": """cross_entropy_logsumexp.html
Write 10 search queries for a course catalog that would find information relevant to the user's interest. The queries should match titles or descriptions of courses in an undergraduate program.

For example, if the student is interested in art, we should query for "art", "photography", "visual rhetoric", etc.

Before responding, think about the user's interest and describe a few specific courses that they might enjoy.

Ensure that each query would match one or more specific courses.
"""},
        {"role": "user", "content": "I want to do graduate school in computational biology. What undergrad courses could I take to prepare myself?"},
    ],
    response_format=SearchTool,
)

event = completion.choices[0].message.parsed
event

SearchTool(thinking="Okay, a graduate program in computational biology. That's a fantastic goal! To help me build a really good list of courses, I need to understand what foundational knowledge is valuable. Computational biology is a broad field – it covers things like statistical modeling, data mining, machine learning, bioinformatics, genomics, proteomics, and more.  Here’s a breakdown of courses that would be particularly useful, grouped by area, and tailored to a graduate-level student aiming for this field - assuming a broad interest in the field, like the ability to learn quickly and tackle complex problems.", queries=['undergraduate courses in statistics', 'undergraduate courses in linear algebra', 'undergraduate courses in programming languages (Python, R)', 'undergraduate courses in bioinformatics', 'undergraduate courses in genomics', 'undergraduate courses in machine learning', 'undergraduate courses in statistical modeling', 'undergraduate courses in data mining', 'undergra

In [19]:
sections_json_url = 'https://app.calvin.edu/coursesearch/AY25FA25_Sections.json'

import requests

sections_json = requests.get(sections_json_url)
sections_json.raise_for_status()
sections = sections_json.json()

In [3]:
len(sections)

1020

In [18]:
next(section for section in sections if section['SectionName'].startswith('CS 108'))
#[section for section in sections if 'programming' in section.get('CourseDescription', '').lower()]

{'AcademicLevel': 'Undergraduate',
 'AcademicPeriod': '2025 Fall (09/02/2025-12/18/2025)',
 'Campus': 'Grand Rapids Campus',
 'CourseNumber': '108',
 'DeliveryMode': 'In-Person',
 'CourseDescription': 'An introduction to computing as a problem-solving discipline. A primary emphasis is on programming as a methodology for problem solving, including: the precise specification of a problem, the design of its solution, the encoding of that solution, and the testing, debugging and maintenance of programs. A secondary emphasis is the discussion of topics from the breadth of computing including historical, theoretical, ethical and biblical perspectives on computing as a discipline. Laboratory. Lab fee - see catalog for details.',
 'SectionEndDate': '2025-12-18',
 'EnrolledCapacity': '8/16',
 'SectionHours': '3',
 'InstructionalFormat': 'Lecture',
 'Instructors': 'Rocky Chang （張蛟川）',
 'Locations': 'Science Building 372 - PC Classroom',
 'MeetingPatterns': 'MWF | 11:00 AM - 12:05 PM | 09/02/2025

In [15]:
course_descriptions = {
    section['SectionName'].split('-', 0)[0].strip(): (section["SectionTitle"], section["CourseDescription"])
    for section in sections
    if "CourseDescription" in section
    and section.get('AcademicLevel') == 'Undergraduate'
    and section.get('Campus') == 'Grand Rapids Campus'
}

In [17]:
len(course_descriptions)

695